# Video Caption 预标注：全 Colab 流程

上传私有 MP4 切片包后，在同一个 Colab 运行时完成：准备输入、切镜/抽帧、ASR、Qwen 视觉预标注、融合、中文翻译和最终标注包下载。API Key 仅从 Colab Secrets 读取。

In [ ]:
import base64, io, shutil, zipfile
from pathlib import Path
SOURCE_BUNDLE_B64 = '...'
repo_dir = Path('/content/video-caption-prelabel')
if repo_dir.exists():
    shutil.rmtree(repo_dir)
repo_dir.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_BUNDLE_B64))) as archive:
    archive.extractall(repo_dir)
%cd {repo_dir}
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install -r requirements-colab.txt


In [ ]:
# 上传 input_videos.zip；可同时上传 source_index.jsonl。
# ZIP 内只放 MP4，不放旧 caption、API Key 或其他敏感文件。
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded))


In [ ]:
import os, shutil, zipfile
from pathlib import Path

archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise RuntimeError('Upload exactly one MP4 ZIP archive.')
media_root = Path('/content/input_videos')
if media_root.exists():
    shutil.rmtree(media_root)
media_root.mkdir(parents=True)
with zipfile.ZipFile(archives[0]) as archive:
    root = media_root.resolve()
    for member in archive.infolist():
        target = (root / member.filename).resolve()
        if target != root and root not in target.parents:
            raise RuntimeError(f'Unsafe ZIP path: {member.filename}')
    archive.extractall(media_root)
source_index = Path('/content/source_index.jsonl') if 'source_index.jsonl' in uploaded else None
print('MP4 count:', len(list(media_root.rglob('*.mp4'))) + len(list(media_root.rglob('*.MP4'))))


In [ ]:
# Colab 左侧 Secrets 中创建 DASHSCOPE_API_KEY，并开启此 Notebook 的访问权限。
from google.colab import userdata
import os
api_key = userdata.get('DASHSCOPE_API_KEY')
if not api_key:
    raise RuntimeError('Missing Colab Secret: DASHSCOPE_API_KEY')
os.environ['DASHSCOPE_API_KEY'] = api_key
print('Secret loaded into this runtime only.')


In [ ]:
# 先运行两条，检查 caption、时间戳和免费 Token 消耗。
import subprocess, sys
run_dir = Path('/content/video_caption_smoke')
cmd = [sys.executable, 'run_colab_pipeline.py', '--media-root', str(media_root),
       '--run-dir', str(run_dir), '--max-items', '2', '--clean-run-dir', '--allow-unresolved']
if source_index:
    cmd += ['--source-index', str(source_index)]
subprocess.run(cmd, check=True)


In [ ]:
import json
summary_path = run_dir / 'delivery' / 'delivery_summary.json'
print(summary_path.read_text(encoding='utf-8'))
print((run_dir / 'fused' / 'fused_preannotations.jsonl').read_text(encoding='utf-8')[:5000])


In [ ]:
# 确认 smoke test 后再运行。此单元会重新处理 20 条，仍只使用 Colab 和免费额度。
full_run_dir = Path('/content/video_caption_full')
cmd = [sys.executable, 'run_colab_pipeline.py', '--media-root', str(media_root),
       '--run-dir', str(full_run_dir), '--max-items', '20', '--clean-run-dir', '--allow-unresolved']
if source_index:
    cmd += ['--source-index', str(source_index)]
subprocess.run(cmd, check=True)


In [ ]:
from google.colab import files
files.download(str(full_run_dir / 'video_caption_delivery.zip'))
